# Assignment 07: Visualization Critique, Redesign, and Explanation

Move through three roles: (1) inspect a bounded pattern, (2) critique and repair a supplied misleading chart, and (3) explain one descriptive finding to a learning-support coordinator. This initial release requires clean local Jupyter or the VS Code notebook interface; Assignment Colab is not yet supported. The course-authored fixtures are synthetic and contain no real or identifying data.

Complete every student `TODO`, restart the kernel, and run all 23 cells from top to bottom. Stored notebook output is not trusted by automation, although freshly rendered charts and explanations remain useful for human review. Regenerate and commit these five separate artifacts: `output/critique_redesign.png`, `output/pathway_explanatory.png`, `output/explanatory_supporting_data.csv`, `output/visualization_evidence.json`, and `output/explanatory_text_alternative.txt`. They must remain visible in VS Code Source Control or GitHub Desktop. Do not edit supplied cells or fixtures.

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import struct
from io import BytesIO
from pathlib import Path

MANIFEST_SHA256 = '1c3397cb2d98ae239f6a7cd254bb3aa9980d94cd23af4546c834a9262de0a28c'
EXPECTED_FIXTURE_SET = 'a07-visualization-v1'
EXPECTED_FILES = {
    'format_completion.csv': {
        'row_grain': 'one row per delivery format and stage',
        'row_count': 4,
        'columns': ['format', 'stage', 'completion_percent'],
        'sha256': '20ad900633154f5f3a2c09cfbc2f890f8423da0897d6345841745332110be66a',
    },
    'pathway_checkpoints.csv': {
        'row_grain': 'one row per learning pathway and checkpoint',
        'row_count': 8,
        'columns': ['pathway', 'checkpoint_number', 'completion_percent'],
        'sha256': 'ec9a336b7fb97418a6f058704f2509c8cee6b13d744efb7a6e3e99224ef8c258',
    },
    'session_observations.csv': {
        'row_grain': 'one row per synthetic learning session',
        'row_count': 12,
        'columns': ['session_id', 'pathway', 'activities_completed', 'reflection_score'],
        'sha256': 'fc4d69ab836288a2fe9c505c65c08413e137e51ab1914cd0e350f6e6636da096',
    },
}

def sha256_bytes(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()

def discover_assignment_root() -> Path:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / '07' / 'assignment'))
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / 'data' / 'fixture.json').is_file():
            return resolved
    raise FileNotFoundError('Could not find data/fixture.json in a standalone or course-repository layout.')

ASSIGNMENT_ROOT = discover_assignment_root()
DATA_DIR = ASSIGNMENT_ROOT / 'data'
OUTPUT_DIR = ASSIGNMENT_ROOT / 'output'
MANIFEST_PATH = DATA_DIR / 'fixture.json'
manifest_bytes = MANIFEST_PATH.read_bytes()
assert sha256_bytes(manifest_bytes) == MANIFEST_SHA256, 'Restore the supplied fixture manifest.'
assert manifest_bytes.endswith(b'\n') and b'\r' not in manifest_bytes, 'fixture.json must use LF and a final newline.'
manifest = json.loads(manifest_bytes.decode('utf-8'))
assert set(manifest) == {'fixture_set_id', 'provenance', 'files'}
assert manifest['fixture_set_id'] == EXPECTED_FIXTURE_SET
assert manifest['provenance'] == 'Course-authored synthetic learning-format, session, and pathway records; no real or identifying data.'
assert isinstance(manifest['files'], list) and len(manifest['files']) == 3
manifest_records = {record['path']: record for record in manifest['files']}
assert set(manifest_records) == set(EXPECTED_FILES)
assert set(path.name for path in DATA_DIR.iterdir() if path.is_file()) == {'fixture.json', *EXPECTED_FILES}
fixture_bytes = {}
for relative_name, expected in EXPECTED_FILES.items():
    record = manifest_records[relative_name]
    assert set(record) == {'path', 'row_grain', 'row_count', 'columns', 'sha256'}
    assert record == {'path': relative_name, **expected}
    relative_path = Path(relative_name)
    assert not relative_path.is_absolute() and relative_path.parts == (relative_name,)
    fixture_path = (DATA_DIR / relative_path).resolve()
    assert fixture_path.parent == DATA_DIR.resolve() and fixture_path.is_file()
    raw = fixture_path.read_bytes()
    assert raw.endswith(b'\n') and b'\r' not in raw
    assert sha256_bytes(raw) == expected['sha256']
    fixture_bytes[relative_name] = raw

OUTPUT_NAMES = (
    'critique_redesign.png',
    'pathway_explanatory.png',
    'explanatory_supporting_data.csv',
    'visualization_evidence.json',
    'explanatory_text_alternative.txt',
)
OUTPUT_DIR.mkdir(exist_ok=True)
for output_name in OUTPUT_NAMES:
    output_path = OUTPUT_DIR / output_name
    if output_path.exists():
        assert output_path.is_file() or output_path.is_symlink(), f'Unexpected output target: {output_name}'
        output_path.unlink()
CRITIQUE_IMAGE_PATH = OUTPUT_DIR / OUTPUT_NAMES[0]
EXPLANATORY_IMAGE_PATH = OUTPUT_DIR / OUTPUT_NAMES[1]
SUPPORTING_DATA_PATH = OUTPUT_DIR / OUTPUT_NAMES[2]
EVIDENCE_JSON_PATH = OUTPUT_DIR / OUTPUT_NAMES[3]
TEXT_ALTERNATIVE_PATH = OUTPUT_DIR / OUTPUT_NAMES[4]

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

assert platform.python_version() == '3.12.13', 'Use CPython 3.12.13.'
assert np.__version__ == '2.0.2'
assert pd.__version__ == '3.0.3'
assert matplotlib.__version__ == '3.10.8'
assert sns.__version__ == '0.13.2'
BLUE = '#0072B2'
ORANGE = '#D55E00'
COURSE_COLORS = [BLUE, ORANGE]

def display_figure(figure) -> None:
    buffer = BytesIO()
    figure.savefig(buffer, format='png', dpi=110, bbox_inches='tight')
    display(Image(data=buffer.getvalue()))

print(f'Assignment root: {ASSIGNMENT_ROOT}')
print(f'Verified fixture set: {EXPECTED_FIXTURE_SET}')

## Vocabulary and prepared data

A **visualization** maps data values to visible properties so a reader can compare them. The **question** is what the chart should help the reader compare or understand. The **audience** is who will use the chart and the context they bring. An **intended claim** is the bounded descriptive conclusion a final chart supports. The **displayed unit** names the quantity reported by an axis or mark. The **grain** states what one row and its corresponding mark or position represents.

Variable roles here are **categorical** (group identity), **quantitative** (numeric magnitude), **ordered** (meaningful sequence), and **identifier** (record identity). An **exploratory visualization** is a truthful view used to inspect a pattern while refining a question. An **explanatory visualization** is a focused chart communicating one selected finding to a named audience.

A **mark** is a visible point, line, or rectangle. An **encoding** maps a value to position, length, color, marker, hatch, or line style. A **redundant encoding** gives the same important category identity a second visual cue. **Visual integrity** means comparisons faithfully represent the data, scale, context, and claim. **Accessibility** means design choices let more readers recover the comparison. A Matplotlib **Figure** is the complete saved canvas; an **Axes** is one plotting area with scales, labels, title, and marks. An **annotation** attaches focused context to a selected mark or position. A **text alternative** names the chart, axes, main pattern, and relevant limitation in text.

The prepared fixtures are complete synthetic rows, not observations from a population:

- `format_completion.csv`: one row per delivery format and stage; categorical format/stage and quantitative prepared completion percent.
- `session_observations.csv`: one row per synthetic learning session; identifier session, categorical pathway, and quantitative activity count/reflection score.
- `pathway_checkpoints.csv`: one row per pathway and checkpoint; categorical pathway, ordered checkpoint, and quantitative prepared completion percent.

These rows can support bounded description only. They do not establish cause, uncertainty, population inference, prediction, or generalization.

In [ ]:
# TODO: Read all three prepared CSVs with pd.read_csv and these explicit dtype maps.
FORMAT_DTYPES = {'format': 'string', 'stage': 'string', 'completion_percent': 'int64'}
SESSION_DTYPES = {'session_id': 'string', 'pathway': 'string', 'activities_completed': 'int64', 'reflection_score': 'int64'}
CHECKPOINT_DTYPES = {'pathway': 'string', 'checkpoint_number': 'int64', 'completion_percent': 'int64'}
format_completion = None
session_observations = None
pathway_checkpoints = None

# TODO: Assert exact columns and shapes: (4, 3), (12, 4), and (8, 3).
# TODO: Save deep copies named format_completion_original, session_observations_original, and pathway_checkpoints_original.
raise NotImplementedError('TODO: load and validate the three prepared fixtures')

## Task 1 — bounded exploration

**TODO:** Before plotting, state what comparison the exploratory question asks for, what one row and point represents, and why the twelve supplied synthetic rows cannot support a causal, inferential, predictive, or generalized conclusion.

In [ ]:
# TODO: Write three nonempty strings consistent with the Markdown contract above.
exploration_question = ''
exploration_observation = ''
exploration_limitation = ''

# TODO: Enter the exact grain and four variable roles defined in the prompt.
exploration_grain = ''
exploration_roles = {}

assert all(isinstance(value, str) and value.strip() for value in [exploration_question, exploration_observation, exploration_limitation])

In [ ]:
def build_exploratory_chart(session_table, pathway_order):
    """Return one exploratory Figure and Axes without saving a file."""
    # TODO: Validate two labels, make one defensive copy, and build exactly one sns.scatterplot.
    # TODO: Map pathway to both hue and style using caller order, course colors, and o/s markers.
    # TODO: Apply the exact labels, title, and Pathway legend; do not mutate the input.
    raise NotImplementedError('TODO: build the exploratory chart')

In [ ]:
# TODO: Call build_exploratory_chart with ['Independent', 'Facilitated'].
# TODO: Verify one Axes, 12 points, two colors, two deduplicated marker shapes, exact labels/title/legend, and source immutability.
# TODO: Assign the returned objects to exploratory_figure and exploratory_axes, then call display_figure(exploratory_figure).
raise NotImplementedError('TODO: run and inspect the exploratory chart without saving a third PNG')

### Task 1 reflection

**TODO:** Explain how the question, one-session grain, variable roles, and visible point marks constrain your observation. Identify why this exploratory view is not an explanatory causal claim.

## Task 2 — critique and redesign

The supplied chart compares prepared completion percentages for Recorded and Live delivery at Start and Finish. Its audience is a learning-support coordinator deciding what questions need follow-up. The displayed unit is prepared completion percent; one row and bar represents one delivery-format/stage combination. `format` and `stage` are categorical, while `completion_percent` is quantitative. The four course-authored rows support only a bounded descriptive comparison—not a causal statement about delivery format. Inspect the five deliberately visible defects before repairing the chart without changing its values.

In [ ]:
flawed_format_order = ['Recorded', 'Live']
flawed_stage_order = ['Start', 'Finish']
flawed_values = {}
for format_label in flawed_format_order:
    for stage_label in flawed_stage_order:
        match = format_completion.loc[
            (format_completion['format'] == format_label) & (format_completion['stage'] == stage_label),
            'completion_percent',
        ]
        assert len(match) == 1
        flawed_values[(format_label, stage_label)] = int(match.iloc[0])

flawed_figure, flawed_axes = plt.subplots(figsize=(7.4, 4.4))
flawed_figure.patch.set_facecolor('#FFF4CC')
flawed_x = np.arange(len(flawed_stage_order))
flawed_width = 0.36
for offset_index, format_label in enumerate(flawed_format_order):
    positions = flawed_x + (offset_index - 0.5) * flawed_width
    heights = [flawed_values[(format_label, stage)] for stage in flawed_stage_order]
    flawed_axes.bar(positions, heights, flawed_width, label=format_label, color=COURSE_COLORS[offset_index])
flawed_axes.set_xticks(flawed_x, flawed_stage_order)
flawed_axes.set_xlabel('Stage')
flawed_axes.set_ylabel('')
flawed_axes.set_title('Live delivery caused stronger completion')
flawed_axes.set_ylim(76, 83)
flawed_axes.grid(True, axis='both', linewidth=2.0, color='#555555')
flawed_axes.legend(title='Delivery format')

assert flawed_axes.get_title() == 'Live delivery caused stronger completion'
assert tuple(round(value) for value in flawed_axes.get_ylim()) == (76, 83)
assert flawed_axes.get_ylabel() == ''
assert {patch.get_facecolor() for patch in flawed_axes.patches} == {matplotlib.colors.to_rgba(BLUE), matplotlib.colors.to_rgba(ORANGE)}
assert flawed_figure.get_facecolor() == matplotlib.colors.to_rgba('#FFF4CC')
assert len(flawed_axes.patches) == 4 and len(flawed_axes.get_xgridlines()) > 0 and len(flawed_axes.get_ygridlines()) > 0
display_figure(flawed_figure)

### Critique the supplied chart

**TODO:** For each of the five defects—unsupported claim, truncated baseline, missing unit, color-only encoding, and distracting decoration—explain how it could mislead or exclude a reader and what a repair must accomplish without changing the prepared values.

In [ ]:
# TODO: Create exactly five dictionaries in the required category order.
# Each dictionary must have only category, problem, and repair keys with nonempty authored text.
critique_entries = []
assert len(critique_entries) == 5

In [ ]:
def build_critique_redesign(summary_table, format_order, stage_order):
    """Return a repaired four-bar Figure and Axes for any valid prepared 2-by-2 table."""
    # TODO: Validate distinct caller labels and a complete, unique 2-by-2 grain; make one defensive copy.
    # TODO: Draw four grouped bars with a zero baseline, exact labels/title, course colors, // and \\ hatches, and value labels.
    # TODO: Remove top/right spines and heavy decoration; place the exact outside frameless legend; do not mutate input.
    raise NotImplementedError('TODO: build the critique redesign')

In [ ]:
# TODO: Call the redesign with canonical format and stage order and assign redesign_figure, redesign_axes.
# TODO: Verify exact 81/77/82/80 bars, zero lower limit, labels/title, four bars, two hatches, four value labels, legend, and immutability.
# TODO: Set the Figure to 7.4 by 4.4 inches, save CRITIQUE_IMAGE_PATH at 150 DPI with bbox_inches='tight', and display it.
raise NotImplementedError('TODO: run, save, and inspect the critique redesign')

## Task 3 — audience-focused explanation

**TODO:** In your own words, state the final question, the learning-support coordinator audience and an actual follow-up use, the bounded intended claim, displayed unit, grain, variable roles, and comparison. Explain why a line chart fits ordered checkpoints and why these prepared pathways do not establish cause.

In [ ]:
# TODO: Write nonempty question, audience, and intended_claim strings matching the Task 3 contract.
question = ''
audience = ''
intended_claim = ''

# TODO: Enter the exact displayed unit, grain, variable roles, and canonical pathway order from the instructions.
displayed_unit = ''
plotting_grain = ''
variable_roles = {}
pathway_order = []
assert all(isinstance(value, str) and value.strip() for value in [question, audience, intended_claim])

In [ ]:
# TODO: Copy the exact three plotted columns into explanatory_supporting_data in canonical row order.
# TODO: Assert schema, eight rows, unique pathway/checkpoint grain, pathway/checkpoint order, and unchanged source.
# TODO: Write SUPPORTING_DATA_PATH as UTF-8 CSV with index=False, lineterminator='\n', and a final newline.
# TODO: Read it back with CHECKPOINT_DTYPES and prove values, dtypes, bytes, and SHA-256 are exact.
explanatory_supporting_data = None
raise NotImplementedError('TODO: validate and export the exact explanatory supporting data')

In [ ]:
def build_explanatory_chart(checkpoint_table, pathway_order):
    """Return a dynamic two-pathway Figure, Axes, and final-gap annotation."""
    # TODO: Validate two distinct labels, complete integer rows, unique grain, and a shared set of at least two checkpoints.
    # TODO: Copy, filter, and sort; draw two paths on an 8 by 4.8 Figure using course colors and redundant o/solid versus s/dashed cues.
    # TODO: Derive labels/order/values, final leader or tie title, absolute-gap text, and annotation target from the input.
    # On a final tie, target the second requested pathway; match annotation text/arrow color to its path.
    # Remove top/right spines and do not mutate input.
    raise NotImplementedError('TODO: build the explanatory chart')

In [ ]:
# TODO: Call the explanatory function with the exact supporting table and pathway_order.
# TODO: Verify two canonical paths, exact title/labels, shared ticks, legend, exact annotation text and (4, 79) target, and immutability.
# TODO: Save EXPLANATORY_IMAGE_PATH at 150 DPI with bbox_inches='tight', replacing stale output, and display it.
raise NotImplementedError('TODO: run, save, and inspect the explanatory chart')

In [ ]:
# TODO: Write one paragraph for explanatory_text_alternative. Name the line chart; both axes and units; both pathways;
# their first-to-last patterns and nine-point final gap; and the descriptive/causal limitation.
explanatory_text_alternative = ''

# TODO: Build visualization_evidence from the named fresh-run variables with the exact required topology and no extra keys.
visualization_evidence = {}

# TODO: Serialize the JSON deterministically as UTF-8 with ensure_ascii=False, indent=2, and exactly one final LF.
# TODO: Write the identical text-alternative value plus exactly one LF with newline='\n'; read both back and compare exactly.
raise NotImplementedError('TODO: export and read back the evidence JSON and text alternative')

## Final human visual review

**TODO:** Answer each item with observable evidence in the exported chart—not only `yes` or `no`.

1. How does the final chart fit the question, coordinator audience, and bounded claim?
2. Why do line marks match the ordered checkpoint comparison?
3. What preserves scale, value, context, and claim integrity?
4. Which redundant encodings and labeling choices improve accessibility?
5. How does the annotation focus attention without overstating the data?
6. How does the text alternative recover the axes, groups, pattern, gap, and limitation?
7. What did you inspect for clipping, overlap, hierarchy, and legibility in the saved PNG?
8. What important limitation remains after the redesign?

In [ ]:
for fixture_name, expected in EXPECTED_FILES.items():
    current = (DATA_DIR / fixture_name).read_bytes()
    assert current == fixture_bytes[fixture_name]
    assert sha256_bytes(current) == expected['sha256']
assert format_completion.equals(format_completion_original)
assert session_observations.equals(session_observations_original)
assert pathway_checkpoints.equals(pathway_checkpoints_original)

assert exploratory_figure.axes == [exploratory_axes]
assert exploratory_axes.get_xlabel() == 'Activities completed (count)'
assert exploratory_axes.get_ylabel() == 'Reflection score (points)'
assert exploratory_axes.get_title() == 'Exploratory view of activities completed and reflection score'
assert redesign_figure.axes == [redesign_axes]
assert redesign_axes.get_ylim()[0] == 0
assert redesign_axes.get_xlabel() == 'Stage' and redesign_axes.get_ylabel() == 'Prepared completion (%)'
assert explanatory_figure.axes == [explanatory_axes]
assert explanatory_axes.get_xlabel() == 'Checkpoint' and explanatory_axes.get_ylabel() == 'Prepared completion (%)'
assert final_gap_annotation in explanatory_axes.texts

def png_dimensions(path: Path) -> tuple[int, int]:
    header = path.read_bytes()[:24]
    assert header[:8] == b'\x89PNG\r\n\x1a\n' and header[12:16] == b'IHDR'
    return struct.unpack('>II', header[16:24])

for image_path in (CRITIQUE_IMAGE_PATH, EXPLANATORY_IMAGE_PATH):
    assert image_path.is_file()
    width, height = png_dimensions(image_path)
    assert 800 <= width <= 2000 and 450 <= height <= 1400
    assert 10_000 <= image_path.stat().st_size <= 2_000_000
assert SUPPORTING_DATA_PATH.read_bytes() == fixture_bytes['pathway_checkpoints.csv']
assert sha256_bytes(SUPPORTING_DATA_PATH.read_bytes()) == EXPECTED_FILES['pathway_checkpoints.csv']['sha256']
assert json.loads(EVIDENCE_JSON_PATH.read_text(encoding='utf-8')) == visualization_evidence
assert EVIDENCE_JSON_PATH.read_bytes().endswith(b'\n') and b'\r' not in EVIDENCE_JSON_PATH.read_bytes()
assert TEXT_ALTERNATIVE_PATH.read_bytes() == (explanatory_text_alternative + '\n').encode('utf-8')
assert set(path.name for path in OUTPUT_DIR.iterdir() if path.is_file()) == {'.gitkeep', *OUTPUT_NAMES}

print('Local machine-readable readiness checks passed for the five regenerated artifacts.')
print('Human visual review and Classroom50 central grading are still required; this cell does not certify clarity, integrity, or accessibility.')
print('Restart the kernel, run all 23 cells, inspect both saved PNGs, then run: python check_assignment.py')